In [1]:
import sys, pandas as pd, numpy as np, sklearn, matplotlib
print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("sklearn   :", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)

Python    : 3.13.9
pandas    : 2.3.3
numpy     : 2.2.3
sklearn   : 1.8.0
matplotlib: 3.10.6


In [2]:
from pathlib import Path

DATA_PATH = Path("../data/raw/APL_Logistics.csv")
print("File exists:", DATA_PATH.exists())

df_raw = pd.read_csv(DATA_PATH, encoding="latin-1")
print("Shape:", df_raw.shape)   # expect (180519, 40)

File exists: True
Shape: (180519, 40)


In [3]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

info = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_missing": df_raw.isna().sum(),
    "n_unique": df_raw.nunique(),
})
info

,dtype,n_missing,n_unique
Type,object,0,4
Days for shipping (real),int64,0,7
Days for shipment (scheduled),int64,0,4
Benefit per order,float64,0,21998
Sales per customer,float64,0,2927
Delivery Status,object,0,4
Late_delivery_risk,int64,0,2
Category Id,int64,0,51
Category Name,object,0,50
Customer City,object,0,563


In [4]:
key_cols = [c for c in df_raw.columns if any(k in c.lower() for k in
            ["days for ship", "late_delivery", "delivery status", "shipping mode",
             "market", "order region", "order country", "customer segment"])]
print(key_cols)
print("Date-like columns:", [c for c in df_raw.columns if "date" in c.lower()])
df_raw[key_cols].head()

['Days for shipping (real)', 'Days for shipment (scheduled)', 'Delivery Status', 'Late_delivery_risk', 'Customer Segment', 'Market', 'Order Country', 'Order Region', 'Shipping Mode']
Date-like columns: []


,Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,Customer Segment,Market,Order Country,Order Region,Shipping Mode
0,6,4,Late delivery,1,Consumer,Pacific Asia,India,South Asia,Standard Class
1,4,4,Shipping on time,0,Consumer,LATAM,Honduras,Central America,Standard Class
2,4,4,Shipping on time,0,Consumer,LATAM,Honduras,Central America,Standard Class
3,6,4,Late delivery,1,Consumer,USCA,Estados Unidos,East of USA,Standard Class
4,6,4,Late delivery,1,Consumer,USCA,Estados Unidos,East of USA,Standard Class


In [5]:
print(df_raw["Late_delivery_risk"].value_counts(dropna=False))
print(df_raw["Late_delivery_risk"].value_counts(normalize=True).round(4))
print("\nFully duplicated rows:", df_raw.duplicated().sum())

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
Late_delivery_risk
1    0.5483
0    0.4517
Name: proportion, dtype: float64

Fully duplicated rows: 0


In [6]:
print(pd.crosstab(df_raw["Delivery Status"], df_raw["Late_delivery_risk"]))
print()
print(pd.crosstab(df_raw["Shipping Mode"], df_raw["Days for shipment (scheduled)"]))

Late_delivery_risk      0      1
Delivery Status                 
Advance shipping    41592      0
Late delivery           0  98977
Shipping canceled    7754      0
Shipping on time    32196      0

Days for shipment (scheduled)     0      1      2       4
Shipping Mode                                            
First Class                       0  27814      0       0
Same Day                       9737      0      0       0
Second Class                      0      0  35216       0
Standard Class                    0      0      0  107752
